# 02 · Laya: zero-shot base vs typed-decisions vs our fine-tune

**Laya** answers *typed questions* about a *state* in a single encoder forward pass (no text generation).
Our three output fields map straight onto its three question types:

| our field | Laya question type |
|---|---|
| `route_target` | `choice` (pick one of 6 teams) |
| `priority_level` | `score` (ordered levels low → critical) |
| `requires_human` | `noul` (yes/no, returns P(true)) |

We compare three Laya arms on the **same test set** as notebook 01:
1. `convaiinnovations/laya`: the general English base, **zero-shot** (never saw our data)
2. `convaiinnovations/laya-typed-decisions`: the authors' fine-tune on synthetic workflows (incl. customer service), **zero-shot** on our data
3. **our fine-tune**: base Laya trained on our train split with `laya.train.finetune` (RLCD loss)

**Runtime:** switch Colab to a **GPU** (Runtime → Change runtime type → T4). Each checkpoint is ~800 MB.

## Setup

Works both on **Colab** and locally. On Colab this clones the repo and moves into it;
locally (opened from `notebooks/`) it just moves up to the repo root so `import src...` works.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/AdityaThakare72/typed-decision-models.git"
BRANCH = "main"

if not os.path.exists("src/dataset.py"):
    if os.path.exists("../src/dataset.py"):      # running locally from notebooks/
        os.chdir("..")
    else:                                         # fresh Colab runtime
        if not os.path.exists("typed-decision-models"):
            subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL], check=True)
        os.chdir("typed-decision-models")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

Install Laya. `USE_TF=0` stops `transformers` from probing TensorFlow (installed on Colab), which can hang model loading.

In [ ]:
%pip install -q laya scikit-learn xgboost pandas
os.environ["USE_TF"] = "0"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1. Load the same splits as notebook 01

`load_splits` reads `data/*.csv`, or rebuilds the identical synthetic data (same seed) if this is a fresh runtime.

In [ ]:
from src.dataset import load_splits, laya_questions, row_to_state

train_df, val_df, test_df = load_splits("data")
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

## 2. What Laya actually sees

The **state** is the ticket as a dict (text + metadata). The **questions** are the same three typed questions for every
ticket; the `criteria` are short descriptions of each option, which is all a zero-shot model has to go on.

In [ ]:
import json

print(json.dumps(row_to_state(test_df.iloc[0]), indent=2))
print(json.dumps(laya_questions(), indent=2))

## 3. Zero-shot: base Laya

`LayaDecisionModel` (in `src/models/laya_model.py`) wraps any checkpoint behind the same `DecisionModel` interface as RF/XGBoost,
so the benchmark code is identical. Let's look at one raw prediction first.

In [ ]:
from src.models.laya_model import LayaDecisionModel, BASE_MODEL_ID, TYPED_DECISIONS_MODEL_ID

laya_base = LayaDecisionModel(BASE_MODEL_ID, device=DEVICE).load()
raw = laya_base.agent.predict(row_to_state(test_df.iloc[0]), laya_questions())
print(json.dumps(raw["answers"], indent=2))
print("true labels:", test_df.iloc[0][["route_target", "priority_level", "requires_human"]].to_dict())

Now the full test set + latency (p50/p95 over 100 single-ticket calls).

In [ ]:
from src.interface import evaluate_model

results = {}
results["laya:laya"] = evaluate_model(laya_base, test_df, n_latency_runs=100)
results["laya:laya"]

## 4. Zero-shot: `laya-typed-decisions`

Same architecture, but already fine-tuned by the authors on synthetic typed-decision workflows. Does that transfer to our
routing schema without any training on our data?

In [ ]:
laya_td = LayaDecisionModel(TYPED_DECISIONS_MODEL_ID, device=DEVICE).load()
results["laya:laya-typed-decisions"] = evaluate_model(laya_td, test_df, n_latency_runs=100)

# free GPU memory before fine-tuning
del laya_td
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 5. Fine-tune base Laya on our tickets

`finetune_laya` writes our rows as Laya JSONL (`{state, questions, expected}`) and calls the package's own trainer
`laya.train.finetune`. Its default loss is **RLCD**: the model is rewarded with a *proper scoring rule* (it scores well only
by putting high probability on the right answer), plus a soft cross-entropy term. After training it refits calibration
temperatures and saves a checkpoint folder that `laya.load` can read.

Knobs:
- `EPOCHS`: 2 is enough for ~2k tickets; raise if val accuracy is still climbing.
- `FREEZE_ENCODER=True`: train only the small decision head. Much faster / lighter, usually a bit less accurate.
- `SAVE_TO_DRIVE`: copy the checkpoint to Google Drive so notebook 03 (a new runtime) can load it.

In [ ]:
from src.models.laya_model import finetune_laya

EPOCHS = 2
FREEZE_ENCODER = False
FINETUNED_DIR = "artifacts/laya-finetuned"

summary = finetune_laya(train_df, FINETUNED_DIR, base_model=BASE_MODEL_ID, val_df=val_df,
                        epochs=EPOCHS, micro_batch=8, grad_accum=4,
                        freeze_encoder=FREEZE_ENCODER, device=DEVICE)
summary

Optional: persist the fine-tuned checkpoint to Google Drive.

In [ ]:
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copytree(FINETUNED_DIR, "/content/drive/MyDrive/typed-decision-models/laya-finetuned", dirs_exist_ok=True)
    print("saved to Drive")

## 6. Evaluate our fine-tuned Laya

Loaded from the local folder, same wrapper, same test set.

In [ ]:
del laya_base
torch.cuda.empty_cache() if torch.cuda.is_available() else None

laya_ft = LayaDecisionModel(FINETUNED_DIR, name="laya:finetuned", device=DEVICE).load()
results["laya:finetuned"] = evaluate_model(laya_ft, test_df, n_latency_runs=100)

## 7. Side by side

`exact_match` is the share of tickets where all three fields are right, which is what an agent router needs.

In [ ]:
import pandas as pd

laya_results = pd.DataFrame(results.values()).set_index("model")
os.makedirs("artifacts", exist_ok=True)
laya_results.to_csv("artifacts/results_laya.csv")
laya_results.round(3)

## 8. Is the confidence trustworthy?

The model card warns Laya is **over-confident**. A quick check: bucket test tickets by the route confidence and compare
it with the actual route accuracy in each bucket. If the model is well calibrated, the two columns should be close.

In [ ]:
from src.interface import outputs_to_frame

outs = laya_ft.predict_batch(test_df)
check = pd.DataFrame({
    "confidence": [o.confidence for o in outs],
    "correct": (outputs_to_frame(outs)["route_target"].values == test_df["route_target"].values),
})
check["bucket"] = pd.cut(check["confidence"], [0, 0.5, 0.7, 0.9, 1.0])
check.groupby("bucket", observed=True).agg(mean_confidence=("confidence", "mean"),
                                           accuracy=("correct", "mean"), n=("correct", "size")).round(3)

**Takeaways to fill in after running:**
- Zero-shot base vs typed-decisions vs fine-tuned exact match: _TODO_
- How much did fine-tuning help priority (the field that depends most on metadata)? _TODO_
- GPU latency p50/p95: _TODO_

Next: `03_full_benchmark_laya_vs_xgboost_rf_bert.ipynb`.